Challenge: Find max temps

In [ ]:
from pyspark import SparkConf, SparkContext
from prettytable import PrettyTable

conf = SparkConf().setMaster('local').setAppName('MinTemps')
sc = SparkContext.getOrCreate(conf)

def parseLine(line):
    fields = line.split(',')
    stationId = fields[0]
    entryType = fields[2]
    temp = float(fields[3])
    return (stationId, entryType, temp)

lines = sc.textFile('../weatherData1800s.csv')
parsedLines = lines.map(parseLine)
maxTemps = parsedLines.filter(lambda x: 'TMAX' in x[1])
stationTemps = maxTemps.map(lambda x: (x[0], x[2]))
maxTemps = stationTemps.reduceByKey(lambda x, y: max(x,y))
results = maxTemps.collect()

table = PrettyTable()
table.field_names = ["Station ID", "Temp"]

for (id, temp) in results:
    table.add_row([id, temp])

print(table)
sc.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/01 10:01:51 WARN Utils: Your hostname, CartersDell, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/01 10:01:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 10:01:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------------+-------+
|  Station ID |  Temp |
+-------------+-------+
| ITE00100554 | 323.0 |
| EZE00100082 | 323.0 |
+-------------+-------+


Min temps, dataframe logic

In [ ]:
from pyspark.sql import SparkSession, functions as func 
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder.appName('Temps').getOrCreate()

schema = StructType([
    StructField('stationId', StringType(), True),
    StructField('date', IntegerType(), True),
    StructField('type', StringType(), True),
    StructField('temp', FloatType(), True)
])

df = spark.read.schema(schema).csv('../weatherData1800s.csv')
df.printSchema()


minTemps = df.filter(df.type == 'TMIN')

stationTemps = minTemps.select('stationId', 'temp')

minTempByStation = stationTemps.groupBy('stationId').min('temp')
minTempByStation.show()

# withColumn returns a new DataFrame by adding a column or replacing the existing column that has the same name.
minTempByStationF = minTempByStation.withColumn('temp',
                                                func.round(func.col('min(temp)') * 0.1 * (9.0 / 5.0) + 32.0, 2))\
                                                .select('stationId', 'temp').sort('temp')

minTempByStationF.show()

root
 |-- stationId: string (nullable = true)
 |-- date: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- temp: float (nullable = true)

+-----------+---------+
|  stationId|min(temp)|
+-----------+---------+
|ITE00100554|   -148.0|
|EZE00100082|   -135.0|
+-----------+---------+

+-----------+----+
|  stationId|temp|
+-----------+----+
|ITE00100554|5.36|
|EZE00100082| 7.7|
+-----------+----+

